In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore") # umap deprecation warnings related to numba do not concern us.


# GRN Construction

Builds the prior knowledge network (PKN) from CollecTRI, using the per-phenotype binarized observation data (`meta`) produced by **"4.5. Phenotype Observations.ipynb"**.

`build_pkn` (`src/grn_inference/pkn.py`, rationale and numbers in `src/grn_inference/README.md`) supports two variants, set with `PKN_CORE` below — both on CollecTRI:

- `None` — the rule of the authors' case study 1 code (Chevalier et al. 2025): every TF -> TF edge of CollecTRI, also between TFs absent from the data, plus TF -> target edges for targets present in the data (`MIN_DETERMINED=0`: no check of binary values).
- `"scc"` — data genes plus the largest strongly connected component of the whole CollecTRI (scBoolSeq approach, Magaña-López et al. 2024). The previous version of this notebook is `PKN_CORE="scc"`, `LARGEST_WCC=True`, `DEFAULT_SIGN_UNKNOWN=False`, `UNSOURCE=False`.

`INCLUDE_UNMEASURED_TFS=False` keeps only data genes (every node constrained by data). `UNSOURCE=True` adds a positive self-loop to nodes without inputs, as `bonesis.InfluenceGraph.from_sif` does in case study 1.

This notebook ends by exporting `{FILE_PREFIX}_prior_grn.json`, `{FILE_PREFIX}_pkn.json`, `{FILE_PREFIX}_data_meta.json` and `{FILE_PREFIX}_config.json`.

The actual BoNesis solving step lives in **"6. BN Inference (BoNesis).ipynb"**, which consumes these exported files. Set the same `FILE_PREFIX` there.

In [ ]:
import sys
import json
from pathlib import Path as path

import pandas as pd

from markers import (
    states_and_markers, # Dictionnary containing phenotypes and their marker genes
    marker_genes, # pandas.Series containing simply the name of markers
)

# Repo root → src on path (shared with the RNA velocity / trajectory-inference notebooks)
ROOT = path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from grn_inference import (
    load_collectri,
    build_pkn,
    in_db,
    df_to_bonesis_data,
    export_pkn_bundle,
)


In [ ]:
FILE_PREFIX = "sum"  # must match FILE_PREFIX used in "4.5. Phenotype Observations.ipynb"
PKN_CORE = None  # None: case study 1 rule (all TF->TF + TF->data genes) | "scc": + largest SCC of the whole CollecTRI (scBoolSeq)
MIN_DETERMINED = 0  # data gene = present in meta (0, as in case study 1) or 0/1 in at least this many phenotypes
INCLUDE_UNMEASURED_TFS = True  # False: only data genes in the PKN
LARGEST_WCC = False  # keep only the largest weakly connected component
DEFAULT_SIGN_UNKNOWN = True  # CollecTRI "default activation" edges -> sign 0 (solver picks)
UNSOURCE = True  # positive self-loop on nodes without inputs (as InfluenceGraph.from_sif in case study 1)

print(
    f"FILE_PREFIX={FILE_PREFIX!r}, PKN_CORE={PKN_CORE!r}, MIN_DETERMINED={MIN_DETERMINED}, "
    f"INCLUDE_UNMEASURED_TFS={INCLUDE_UNMEASURED_TFS}, LARGEST_WCC={LARGEST_WCC}, "
    f"DEFAULT_SIGN_UNKNOWN={DEFAULT_SIGN_UNKNOWN}, UNSOURCE={UNSOURCE}"
)


### TF database loading

In [ ]:
net = load_collectri()


In [ ]:
meta = pd.read_csv(f"meta_{FILE_PREFIX}_bin_aggregated.csv", index_col=0)
with open(f"{FILE_PREFIX}_config.json") as _cfg:
    INCLUDE_HC_CONSTRAINT = json.load(_cfg)["INCLUDE_HC_CONSTRAINT"]

meta_var = meta
print(meta.shape, f"INCLUDE_HC_CONSTRAINT={INCLUDE_HC_CONSTRAINT}")
meta.iloc[:, :10]


In [ ]:
# Observation diagnostics: undetermined (NaN) fraction per phenotype, and genes
# determined in every phenotype with the same value (invariant)
print("NaN fraction per phenotype:")
print(meta.T.isna().mean().round(3).to_string())
invariant = meta.columns[meta.var(skipna=False) == 0]
print(f"\ninvariant genes: {len(invariant)} (in CollecTRI: {in_db(meta[invariant], net).sum()})")


### PKN

In [ ]:
pkn_graph, pkn_report = build_pkn(
    net,
    meta,
    core=PKN_CORE,
    min_determined=MIN_DETERMINED,
    include_unmeasured_tfs=INCLUDE_UNMEASURED_TFS,
    largest_wcc=LARGEST_WCC,
    default_sign_unknown=DEFAULT_SIGN_UNKNOWN,
)
pkn_report


In [ ]:
print(f"markers in PKN: {marker_genes.isin(pkn_graph).mean():.2%}")
print("\t".join(["Cell", "pObserved", "pPKN"]))
_observed = meta.columns[meta.notna().any()]
for pheno, genes in states_and_markers.items():
    if pheno in meta.index:
        _genes = pd.Index(genes)
        print(f"{pheno}\t{100 * _genes.isin(_observed).mean():.2f}\t{100 * _genes.isin(pkn_graph).mean():.2f}")


In [ ]:
import bonesis


In [ ]:
data_meta = df_to_bonesis_data(meta[[g for g in meta.columns if g in pkn_graph]])
data_meta.keys()


In [ ]:
pkn = bonesis.domains.InfluenceGraph(
    pkn_graph, maxclause=8, allow_skipping_nodes=True, canonic=False
)
if UNSOURCE:
    _sources = pkn.sources()
    pkn.unsource()
    print(f"unsource: positive self-loop added on {len(_sources)} nodes without inputs")
print(f"PKN: nodes={len(pkn)}, edges={pkn.number_of_edges()}")


In [ ]:
bo = bonesis.BoNesis(pkn, data_meta)
bo


In [ ]:
# prior_grn == the measured PKN itself (no larger CollecTRI core around it any more)
export_pkn_bundle(FILE_PREFIX, pkn_graph, pkn, data_meta, INCLUDE_HC_CONSTRAINT)

print(
    f"Exported pkn ({len(pkn)} nodes), data_meta ({len(data_meta)} observations) "
    f"and config for FILE_PREFIX={FILE_PREFIX!r}"
)


### Next step

Continue in **"6. BN Inference (BoNesis).ipynb"** — set `FILE_PREFIX` there to the same value as `FILE_PREFIX` above, to load the files exported here.